# 📖 Tutorial 01: Understanding the Game & Environment API

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lessen-xu/Hybrid-Chess/blob/main/notebooks/01_game_rules_and_env.ipynb)

---

## Who is this for?

This tutorial is written for **complete beginners** — you don't need any prior knowledge of reinforcement learning, chess, or Chinese chess. By the end, you'll understand:

1. What Hybrid Chess is and why it's an interesting RL problem
2. How game environments work in RL (the "agent-environment loop")
3. How to use the Hybrid Chess environment API
4. How to customize game rules — and why that matters for research

**Prerequisites:** Basic Python (variables, loops, functions, classes).

---

## 0. Setup

If you're running this on **Google Colab**, uncomment the line below to install the package.
If you've already cloned the repo locally, you can skip this.

In [ ]:
# Uncomment for Google Colab:
# !pip install -q git+https://github.com/lessen-xu/Hybrid-Chess.git

In [ ]:
# Verify installation
import hybrid
print(f"✅ Hybrid Chess loaded from: {hybrid.__file__}")

---

## 1. The Game: International Chess vs Chinese Chess

### 1.1 Background

**International Chess** (国际象棋) and **Chinese Chess / Xiangqi** (中国象棋) are two of the world's most popular board games. They share a common ancestor but evolved very differently:

| | International Chess | Chinese Chess (Xiangqi) |
|---|---|---|
| **Board** | 8×8 | 9×10, with a river |
| **Strongest piece** | Queen (moves any direction, any distance) | Chariot (= Rook, orthogonal only) |
| **Unique piece** | Bishop (diagonal slide) | Cannon (slides to move, **jumps** to capture) |
| **King** | Moves anywhere on the board | General: confined to a 3×3 "palace" |
| **Stalemate** | Draw! | **Loss** for the stalemated side |

### 1.2 Hybrid Chess: What happens when they fight?

**Hybrid Chess** puts them on the **same board** — a 9×10 grid (Xiangqi's board dimensions). Each side uses its own piece movement rules:

```
     Xiangqi Side (top)
  ┌─────────────────────────────┐
  │  c  h  e  a  g  a  e  h  c  │  Row 10 (y=9): back rank
  │  .  .  .  .  .  .  .  .  .  │  Row 9  (y=8)
  │  .  n  .  .  .  .  .  n  .  │  Row 8  (y=7): cannons
  │  s  .  s  .  s  .  s  .  s  │  Row 7  (y=6): soldiers
  │  ─  ─  ─  ─  ─  ─  ─  ─  ─ │  ← River (y=5)
  │  .  .  .  .  .  .  .  .  .  │  Row 5  (y=4)
  │  .  .  .  .  .  .  .  .  .  │  Row 4  (y=3)
  │  .  .  .  .  .  .  .  .  .  │  Row 3  (y=2)
  │  P  P  P  P  P  P  P  P  P  │  Row 2  (y=1): pawns
  │  R  N  B  Q  K  B  N  R  .  │  Row 1  (y=0): back rank
  └─────────────────────────────┘
     Chess Side (bottom)
     a  b  c  d  e  f  g  h  i
```

### 1.3 Why is this interesting for AI?

Most game AI research is done on **symmetric** games — both players have the same pieces and the same rules. Hybrid Chess is **asymmetric**: the two sides play fundamentally different games. This creates unique challenges:

- **No known openings or strategies** — unlike standard chess, no one has studied this game for centuries
- **Balance is a real question** — is the Queen side stronger? Or does the Cannon's jump-capture compensate?
- **The AI must learn everything from scratch** — no books, no databases, pure self-play

This makes it an ideal testbed for **reinforcement learning algorithms** like AlphaZero.

### 1.4 Piece Reference

Here are all the pieces in the game. Don't worry about memorizing them — you'll pick them up as you play.

#### Chess Side (bottom, uppercase letters)

| Symbol | Piece | Movement |
|--------|-------|----------|
| `K` | King | 1 step in any direction (8 neighbors) |
| `Q` | Queen | Slides any distance in any direction (the most powerful piece!) |
| `R` | Rook (×2) | Slides orthogonally (horizontal/vertical), any distance |
| `B` | Bishop (×2) | Slides diagonally, any distance |
| `N` | Knight (×2) | "L-shape" jump: 2+1 squares. **No** leg-blocking |
| `P` | Pawn (×9) | Forward 1 (or 2 from starting row). Captures diagonally. Promotes at top row |

#### Xiangqi Side (top, lowercase letters)

| Symbol | Piece | Movement |
|--------|-------|----------|
| `g` | General (將) | 1 step orthogonally, **confined to 3×3 palace** |
| `a` | Advisor (士, ×2) | 1 step diagonally, **confined to palace** |
| `e` | Elephant (象, ×2) | 2-step diagonal jump, **cannot cross the river**, blocked if piece in the way |
| `h` | Horse (馬, ×2) | Same L-shape as Knight, BUT **with** leg-blocking |
| `c` | Chariot (車, ×2) | Identical to Chess Rook (orthogonal slide) |
| `n` | Cannon (砲, ×2) | Slides orthogonally to **move**, but must **jump over exactly 1 piece** to capture |
| `s` | Soldier (卒, ×5) | Forward only before river; forward + sideways after crossing |

#### Special Rule: Flying General

If the General (`g`) and King (`K`) are on the **same column** with no pieces between them, the General can capture the King directly — like a laser shot down the file. This forces the Chess side to always keep something between them.

---

## 2. The Agent-Environment Loop

### 2.1 Theory: How RL works

Before we touch any code, let's understand the fundamental concept behind all reinforcement learning.

In RL, we have two entities:
- **The Environment** — the game world. It knows the rules, the current state, and what happens when you take an action.
- **The Agent** — the decision-maker. It observes the state and chooses an action.

They interact in a loop:

```
                    ┌──────────────┐
                    │              │
         state      │  Environment │     reward
    ┌──────────────▶│   (Game)     │──────────────┐
    │               │              │              │
    │               └──────┬───────┘              │
    │                      │                      │
    │                action│                      │
    │                      │                      ▼
    │               ┌──────┴───────┐              │
    │               │              │              │
    └───────────────│    Agent     │◀─────────────┘
                    │  (AI / You)  │
                    │              │
                    └──────────────┘
```

Each step of the loop:
1. The **environment** provides the current **state** (the board position)
2. The **agent** observes the state and chooses an **action** (a move)
3. The **environment** applies the action, moves to a new state, and returns:
   - The **new state**
   - A **reward** (+1 for winning, -1 for losing, 0 otherwise)
   - Whether the game is **done** (terminated)

### 2.2 In code: The Environment API

Our environment is the `HybridChessEnv` class. It provides exactly these methods:

```python
env = HybridChessEnv()  # Create the environment
state = env.reset()     # Start a new game → returns initial state

# The loop:
legal_moves = env.legal_moves()           # What actions are available?
state, reward, done, info = env.step(move) # Take an action → get result
```

Let's try it!

In [ ]:
from hybrid.core.env import HybridChessEnv
from hybrid.core.render import render_board
from hybrid.core.types import Side, Move

# Step 1: Create the environment
env = HybridChessEnv()

# Step 2: Reset (start a new game)
state = env.reset()

# Let's look at what 'state' contains:
print("📋 GameState contents:")
print(f"  • side_to_move: {state.side_to_move}  (who plays next)")
print(f"  • ply:          {state.ply}  (half-move count, starts at 0)")
print(f"  • board:        9×10 grid of pieces")
print()
print("🎮 Initial board:")
print(render_board(state.board))

### 2.3 Understanding the state

The `GameState` object contains everything we need to know about the current game position:

- **`board`**: A 9×10 grid. Each cell is either `None` (empty) or a `Piece` object.
- **`side_to_move`**: Either `Side.CHESS` or `Side.XIANGQI` — whose turn it is.
- **`ply`**: The half-move counter. In chess, "ply" means one player's move. So after Chess moves once & Xiangqi moves once, ply = 2.

Let's inspect the board more closely:

In [ ]:
# The board is a grid. We can look at individual squares.
# board.get(x, y) returns a Piece or None.
# x = column (0-8, left to right: a-i)
# y = row    (0-9, bottom to top)

board = state.board

# Let's see what's at a few squares:
examples = [
    (4, 0, "e1 — Chess King's starting square"),
    (3, 0, "d1 — Chess Queen's starting square"),
    (4, 9, "e10 — Xiangqi General's starting square"),
    (1, 7, "b8 — Xiangqi Cannon's starting square"),
    (4, 4, "e5 — An empty square in the middle"),
]

for x, y, desc in examples:
    piece = board.get(x, y)
    if piece:
        print(f"  ({x},{y}) {desc}: {piece.side.name}'s {piece.kind.name}")
    else:
        print(f"  ({x},{y}) {desc}: empty")

In [ ]:
# We can also iterate over all pieces on the board:
# board.iter_pieces() yields (x, y, piece) for every occupied square

chess_pieces = []
xiangqi_pieces = []

for x, y, piece in state.board.iter_pieces():
    if piece.side == Side.CHESS:
        chess_pieces.append(piece.kind.name)
    else:
        xiangqi_pieces.append(piece.kind.name)

print(f"Chess side has {len(chess_pieces)} pieces:")
# Count each piece type
from collections import Counter
for kind, count in sorted(Counter(chess_pieces).items()):
    print(f"  {kind}: {count}")

print(f"\nXiangqi side has {len(xiangqi_pieces)} pieces:")
for kind, count in sorted(Counter(xiangqi_pieces).items()):
    print(f"  {kind}: {count}")

---

## 3. Legal Moves and Taking Actions

### 3.1 Theory: Action spaces in board games

In reinforcement learning, the **action space** is the set of all possible actions the agent can take. In board games, this means the set of all legal moves.

A key challenge: the number of legal moves **changes** every turn. In the opening, Chess might have ~30 legal moves, but in a complex middlegame it could have 50+. This is called a **variable-size action space**, and it's one of the things that makes board games tricky for RL.

### 3.2 The Move object

Each move is represented by a `Move` dataclass:

```python
Move(fx, fy, tx, ty, promotion)
```

- `fx, fy` — **from** square (where the piece is now)
- `tx, ty` — **to** square (where the piece moves to)
- `promotion` — only for Pawn promotion (when a Pawn reaches the top row, it becomes a Queen, Rook, Bishop, or Knight)

Let's see this in action:

In [ ]:
# Get all legal moves for the current side (Chess goes first)
legal_moves = env.legal_moves()

print(f"🎯 {len(legal_moves)} legal moves for {state.side_to_move.name}")
print()

# Group moves by piece type for clarity
moves_by_piece = {}
for mv in legal_moves:
    piece = state.board.get(mv.fx, mv.fy)
    kind = piece.kind.name
    if kind not in moves_by_piece:
        moves_by_piece[kind] = []
    moves_by_piece[kind].append(mv)

for kind, moves in moves_by_piece.items():
    print(f"  {kind} ({len(moves)} moves):")
    for mv in moves[:3]:  # Show first 3 of each type
        print(f"    ({mv.fx},{mv.fy}) → ({mv.tx},{mv.ty})")
    if len(moves) > 3:
        print(f"    ... and {len(moves) - 3} more")

### 3.3 Taking a step

The `env.step(move)` method is the core of the agent-environment loop. It:

1. Validates the move is legal
2. Applies the move to the board
3. Checks if the game is over (checkmate, stalemate, etc.)
4. Returns a tuple of `(new_state, reward, done, info)`

Let's make a few moves and see what happens:

In [ ]:
import random
random.seed(42)

# Start fresh
env = HybridChessEnv()
state = env.reset()

print("Initial position:")
print(render_board(state.board))

# Play 4 moves (2 per side)
for i in range(4):
    # Who is playing?
    side = state.side_to_move
    
    # Get legal moves and pick one randomly
    legal = env.legal_moves()
    move = random.choice(legal)
    
    # What piece is moving?
    piece = state.board.get(move.fx, move.fy)
    # Is it a capture?
    target = state.board.get(move.tx, move.ty)
    capture_str = f" captures {target.kind.name}!" if target else ""
    
    # Execute the move
    state, reward, done, info = env.step(move)
    
    print(f"\n{'─'*40}")
    print(f"Move {i+1}: {side.name}'s {piece.kind.name} "
          f"({move.fx},{move.fy})→({move.tx},{move.ty}){capture_str}")
    print(f"  reward={reward}, done={done}, ply={state.ply}")
    print(render_board(state.board))
    
    if done:
        print(f"\n🏁 Game over! Reason: {info.reason}")
        print(f"   Winner: {info.winner.name if info.winner else 'Draw'}")
        break

### 3.4 Understanding rewards

The reward system is simple but important:

| Situation | Reward |
|-----------|--------|
| Game ongoing (no one wins yet) | `0.0` |
| The moving side wins | `+1.0` |
| The moving side loses | `-1.0` |
| Draw (repetition, max moves) | `0.0` |

> ⚠️ **Important**: The reward is always from the **moving side's perspective**. If Chess makes a move that results in a Xiangqi win, Chess gets reward `-1.0`.

This is called a **sparse reward** — the agent only gets a non-zero signal at the very end of the game. One of the hardest challenges in RL is learning from sparse rewards over very long episodes (a game can last hundreds of moves).

---

## 4. Playing a Complete Game

### 4.1 The simplest "agent": Random play

The most basic agent just picks a random legal move every turn. This is useful as:
- A **baseline** — any real agent should beat random play
- A **smoke test** — if random play crashes, something is wrong with the environment
- A way to understand **game dynamics** — how long do random games last? Who tends to win?

Let's play 50 random games and collect statistics:

In [ ]:
def play_random_game(seed=0):
    """Play a complete game with both sides choosing randomly.
    
    Returns a dict with game statistics.
    """
    rng = random.Random(seed)
    env = HybridChessEnv()
    state = env.reset()
    info = None
    
    while True:
        legal = env.legal_moves()
        if not legal:
            break
        move = rng.choice(legal)
        state, reward, done, info = env.step(move)
        if done:
            break
    
    return {
        "ply": state.ply,
        "winner": info.winner.name if (info and info.winner) else "Draw",
        "reason": info.reason if info else "no legal moves",
    }

# Play 50 random games
print("Playing 50 random games...")
results = [play_random_game(seed=i) for i in range(50)]

# Summarize
chess_wins = sum(1 for r in results if r["winner"] == "CHESS")
xiangqi_wins = sum(1 for r in results if r["winner"] == "XIANGQI")
draws = sum(1 for r in results if r["winner"] == "Draw")
avg_ply = sum(r["ply"] for r in results) / len(results)

print(f"\n📊 Results (50 random games):")
print(f"  ♔ Chess wins:    {chess_wins:>3} ({chess_wins/50*100:.0f}%)")
print(f"  將 Xiangqi wins: {xiangqi_wins:>3} ({xiangqi_wins/50*100:.0f}%)")
print(f"  🤝 Draws:        {draws:>3} ({draws/50*100:.0f}%)")
print(f"  📏 Avg game length: {avg_ply:.0f} half-moves")

# Termination reasons
print(f"\n🏁 How games ended:")
from collections import Counter
for reason, count in Counter(r["reason"] for r in results).most_common():
    print(f"  {reason}: {count}")

### 4.2 What do we learn from random play?

You'll likely see that random games are very chaotic — pieces get captured randomly, and games often end by the 400-move limit or by one side accidentally checkmating the other.

Key observations:
- **Random play is NOT balanced** — one side probably wins more often. This tells us something about the inherent asymmetry.
- **Games are long** — random agents don't know how to finish games efficiently.
- **This is why we need RL** — a trained agent should win faster, more consistently, and develop real strategies.

---

## 5. Custom Rule Variants with VariantConfig

### 5.1 Why rule variants matter

In asymmetric games, **balance** is a central question. If one side always wins, the game isn't fun (or useful for research). In traditional game design, you balance by tweaking the rules. In RL research, this is called **ablation study** — you systematically change one thing at a time to understand its effect.

Our `VariantConfig` system lets you do this **without modifying any source code**. It's a frozen (immutable) dataclass that controls:
- Which pieces appear on the board
- Which special rules are active

### 5.2 How VariantConfig works

Under the hood, `VariantConfig` is a frozen Python dataclass:

```python
@dataclass(frozen=True)  # frozen = you can't modify it after creation
class VariantConfig:
    no_queen: bool = False          # Remove Chess's Queen?
    extra_cannon: bool = False      # Give Xiangqi a 3rd Cannon?
    flying_general: bool = True     # Enable the flying-general rule?
    # ... more options
```

Why frozen? Because in parallel self-play training, multiple game instances run at the same time. If the config were mutable, one process could accidentally change another's settings. Frozen = thread-safe.

Let's experiment:

In [ ]:
from hybrid.core.config import VariantConfig

# First, let's see all available options:
print("📋 All VariantConfig options:")
print()
for name, field in VariantConfig.__dataclass_fields__.items():
    print(f"  {name:25s} (default: {field.default})")

In [ ]:
# Compare: Standard vs No Queen
# Removing the Queen is the most drastic nerf to the Chess side.

standard_env = HybridChessEnv()  # default = VariantConfig()
noqueen_env = HybridChessEnv(variant=VariantConfig(no_queen=True))

std_state = standard_env.reset()
nq_state = noqueen_env.reset()

print("=== Standard (with Queen at d1) ===")
print(render_board(std_state.board))
print(f"Legal moves: {len(standard_env.legal_moves())}")

print("\n=== No Queen (d1 is empty!) ===")
print(render_board(nq_state.board))
print(f"Legal moves: {len(noqueen_env.legal_moves())}")

print(f"\n💡 Removing the Queen reduced Chess's opening moves by "
      f"{len(standard_env.legal_moves()) - len(noqueen_env.legal_moves())}")

In [ ]:
# Let's test balance: does removing the Queen change who wins in random play?

def test_balance(variant, n_games=50, label=""):
    """Run n_games random games and return win rates."""
    chess_w = 0
    for i in range(n_games):
        rng = random.Random(i + 9999)  # different seeds than before
        env = HybridChessEnv(variant=variant)
        state = env.reset()
        info = None
        while True:
            legal = env.legal_moves()
            if not legal:
                break
            state, reward, done, info = env.step(rng.choice(legal))
            if done:
                break
        if info and info.winner == Side.CHESS:
            chess_w += 1
    return chess_w / n_games

# Test several variants
variants = {
    "Standard": VariantConfig(),
    "No Queen": VariantConfig(no_queen=True),
    "Extra Cannon": VariantConfig(extra_cannon=True),
    "No Queen + Extra Cannon": VariantConfig(no_queen=True, extra_cannon=True),
}

print(f"{'Variant':<30} {'Chess Win Rate':>15}")
print("─" * 47)
for name, vc in variants.items():
    rate = test_balance(vc, n_games=30)
    bar = '█' * int(rate * 20) + '░' * (20 - int(rate * 20))
    print(f"{name:<30} {bar} {rate*100:5.1f}%")

### 5.3 Serialization

When training RL agents, you often need to save and load configurations (for checkpoints, experiment tracking, etc.).

`VariantConfig` supports this natively:

In [ ]:
import json

# Create a variant
my_variant = VariantConfig(no_queen=True, extra_cannon=True)

# Serialize to dict (for JSON, checkpoints, etc.)
d = my_variant.to_dict()
print("Serialized:")
print(json.dumps(d, indent=2))

# Reconstruct from dict
reconstructed = VariantConfig.from_dict(d)
print(f"\nRound-trip check: {my_variant == reconstructed}")

# This means your experiment config is always saved alongside model checkpoints!
print("\n✅ Configs survive serialization — great for reproducible experiments.")

---

## 6. FEN Notation: Loading Arbitrary Positions

### 6.1 What is FEN?

**FEN** (Forsyth-Edwards Notation) is a standard way to describe a chess position as a text string. In standard chess it looks like:

```
rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w
```

We use a similar notation adapted for Hybrid Chess:
- Each row is separated by `/`
- Numbers represent consecutive empty squares
- Letters represent pieces (uppercase = Chess, lowercase = Xiangqi)
- A final `c` or `x` indicates side to move (Chess or Xiangqi)

### 6.2 Why is FEN useful for RL?

- **Curriculum learning**: Start training from endgame positions (fewer pieces = simpler to learn)
- **Debugging**: Reproduce a specific position to test your agent
- **Evaluation**: Test your agent on specific puzzles or scenarios

In [ ]:
from hybrid.core.fen import parse_fen, board_to_fen

# Create a simple endgame: just a King vs General with one Pawn
endgame_fen = "4g4/9/9/9/9/9/9/9/4P4/4K4 c"

board, side = parse_fen(endgame_fen)

print(f"FEN: {endgame_fen}")
print(f"Side to move: {side.name}")
print()
print(render_board(board))

# Load into environment and check legal moves
env = HybridChessEnv()
state = env.reset_from_fen(endgame_fen)
print(f"\n{len(env.legal_moves())} legal moves from this position")
print("\n💡 This is great for curriculum learning: start with simple"
      " endgames, then gradually increase complexity!")

In [ ]:
# FEN round-trip: position → string → position
env = HybridChessEnv()
state = env.reset()

# Convert the starting position to FEN
fen = board_to_fen(state.board, state.side_to_move)
print(f"Starting position as FEN:\n{fen}")

# Verify round-trip is lossless
board2, side2 = parse_fen(fen)
fen2 = board_to_fen(board2, side2)
assert fen == fen2
print("\n✅ Round-trip verified: FEN → Board → FEN gives same string")

---

## 7. AI Agents: From Random to Smart

### 7.1 The Agent abstraction

All agents in Hybrid Chess implement a simple interface:

```python
class Agent(ABC):
    name: str = "agent"

    @abstractmethod
    def select_move(self, state: GameState, legal_moves: list[Move]) -> Move:
        ...  # Return the move you want to play
```

The environment provides the state and the legal moves. The agent's only job is to pick one.

### 7.2 Built-in agents

The project comes with several agents of increasing strength:

| Agent | Strategy | Strength |
|-------|----------|----------|
| `RandomAgent` | Picks uniformly at random | Terrible (baseline) |
| `GreedyAgent` | Always captures if possible | Slightly better |
| `AlphaBetaAgent` | Minimax search + hand-crafted evaluation | Decent (no learning!) |
| `AlphaZeroMiniAgent` | MCTS + neural network | Best (after training) |

Let's pit them against each other:

In [ ]:
from hybrid.agents.random_agent import RandomAgent
from hybrid.agents.greedy_agent import GreedyAgent

def play_match(chess_agent, xiangqi_agent, n_games=10):
    """Play N games and count results."""
    results = {"chess": 0, "xiangqi": 0, "draw": 0}
    
    for i in range(n_games):
        env = HybridChessEnv()
        state = env.reset()
        agents = {Side.CHESS: chess_agent, Side.XIANGQI: xiangqi_agent}
        info = None
        
        while True:
            legal = env.legal_moves()
            if not legal:
                break
            agent = agents[state.side_to_move]
            move = agent.select_move(state, legal)
            state, reward, done, info = env.step(move)
            if done:
                break
        
        if info and info.winner == Side.CHESS:
            results["chess"] += 1
        elif info and info.winner == Side.XIANGQI:
            results["xiangqi"] += 1
        else:
            results["draw"] += 1
    
    return results

# Random vs Random
r1 = play_match(RandomAgent(), RandomAgent(), n_games=10)
print(f"Random vs Random:      Chess {r1['chess']}, Xiangqi {r1['xiangqi']}, Draw {r1['draw']}")

# Random vs Greedy
r2 = play_match(RandomAgent(), GreedyAgent(), n_games=10)
print(f"Random vs Greedy:      Chess {r2['chess']}, Xiangqi {r2['xiangqi']}, Draw {r2['draw']}")

# Greedy vs Random
r3 = play_match(GreedyAgent(), RandomAgent(), n_games=10)
print(f"Greedy vs Random:      Chess {r3['chess']}, Xiangqi {r3['xiangqi']}, Draw {r3['draw']}")

# Greedy vs Greedy
r4 = play_match(GreedyAgent(), GreedyAgent(), n_games=10)
print(f"Greedy vs Greedy:      Chess {r4['chess']}, Xiangqi {r4['xiangqi']}, Draw {r4['draw']}")

print("\n💡 Even simple heuristics (capture when possible) significantly")
print("   outperform random play. Imagine what a trained neural net can do!")

### 7.3 Writing your own agent

Creating a custom agent is as simple as subclassing `Agent` and implementing `select_move`.

Here's a slightly smarter agent — it prioritizes capturing high-value pieces:

In [ ]:
from hybrid.agents.base import Agent
from hybrid.core.env import GameState
from hybrid.core.types import Move, PieceKind
from typing import List

# Piece values — how much is each piece "worth"?
PIECE_VALUES = {
    PieceKind.PAWN: 1, PieceKind.SOLDIER: 1,
    PieceKind.KNIGHT: 3, PieceKind.HORSE: 3,
    PieceKind.BISHOP: 3, PieceKind.ELEPHANT: 2, PieceKind.ADVISOR: 2,
    PieceKind.ROOK: 5, PieceKind.CHARIOT: 5,
    PieceKind.CANNON: 4,
    PieceKind.QUEEN: 9,
    PieceKind.KING: 100, PieceKind.GENERAL: 100,  # invaluable!
}

class SmartCaptureAgent(Agent):
    """Captures the highest-value piece available. Otherwise picks randomly."""
    name = "smart_capture"
    
    def select_move(self, state: GameState, legal_moves: List[Move]) -> Move:
        best_move = None
        best_value = -1
        
        for mv in legal_moves:
            target = state.board.get(mv.tx, mv.ty)
            if target is not None:
                value = PIECE_VALUES.get(target.kind, 0)
                if value > best_value:
                    best_value = value
                    best_move = mv
        
        if best_move is not None:
            return best_move
        return random.choice(legal_moves)

# Test it against the built-in agents
r5 = play_match(SmartCaptureAgent(), RandomAgent(), n_games=10)
print(f"SmartCapture vs Random: Chess {r5['chess']}, Xiangqi {r5['xiangqi']}, Draw {r5['draw']}")

r6 = play_match(SmartCaptureAgent(), GreedyAgent(), n_games=10)
print(f"SmartCapture vs Greedy: Chess {r6['chess']}, Xiangqi {r6['xiangqi']}, Draw {r6['draw']}")

print("\n💡 Even this simple heuristic (capture the most valuable piece)")
print("   can be quite effective. But it still lacks planning and strategy.")
print("   That's where search algorithms (AlphaBeta) and learning (AlphaZero) come in!")

---

## 8. The Gymnasium Interface

### 8.1 What is Gymnasium?

[Gymnasium](https://gymnasium.farama.org/) (formerly OpenAI Gym) is the **standard API** for RL environments. Almost every RL library (Stable-Baselines3, CleanRL, RLlib, etc.) uses this interface. By providing a Gymnasium wrapper, our game can plug into any of these frameworks.

The Gymnasium API is slightly different from our native API:

| | Native API | Gymnasium API |
|---|---|---|
| **Observation** | `GameState` object | `numpy.ndarray` (14×10×9 tensor) |
| **Action** | `Move` object | `int` (index 0–8279) |
| **Reset** | `state = env.reset()` | `obs, info = env.reset()` |
| **Step** | `state, reward, done, info = env.step(move)` | `obs, reward, terminated, truncated, info = env.step(action)` |

### 8.2 Why 8,280 actions?

The action space needs a fixed size (neural networks need fixed-size inputs and outputs). We encode every possible move as:

```
92 move planes × 10 rows × 9 cols = 8,280 possible actions

The 92 planes cover:
  - 72 slide planes: 8 directions × 9 distances
  - 8 knight planes: 8 L-shaped jumps
  - 12 promotion planes: 3 directions × 4 piece types
```

Most of these are illegal in any given position. That's why `info["legal_actions"]` tells you which ones are valid.

In [ ]:
try:
    import gymnasium as gym
    import hybrid.gym_env  # registers HybridChess-v0
    
    env = gym.make("HybridChess-v0")
    obs, info = env.reset()
    
    print(f"Observation shape: {obs.shape}")
    print(f"  → 14 channels × 10 rows × 9 columns")
    print(f"  → 13 piece-type planes + 1 side-to-move plane")
    print(f"\nAction space: {env.action_space}")
    print(f"  → 8,280 possible actions (most are illegal in any position)")
    print(f"\nLegal actions right now: {len(info['legal_actions'])} out of 8,280")
    print(f"Side to move: {info['side_to_move']}")
    
    # Take a legal action
    action = info["legal_actions"][0]
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"\nAfter one move:")
    print(f"  Reward: {reward}")
    print(f"  Terminated: {terminated}")
    print(f"  Side to move: {info['side_to_move']}")
    print(f"  Legal actions: {len(info['legal_actions'])}")
    
    print("\n✅ Ready to plug into Stable-Baselines3, CleanRL, RLlib, etc.!")
    
except ImportError:
    print("⚠️ gymnasium not installed.")
    print("   Install with: pip install gymnasium")
    print("   The core environment works fine without it!")

---

## 🎓 Summary: What We Learned

| Concept | What We Did |
|---------|-------------|
| **The Game** | Hybrid Chess: Chess vs Xiangqi on a shared 9×10 board |
| **Agent-Environment Loop** | `reset()` → `legal_moves()` → `step(move)` → repeat |
| **GameState** | Contains `board`, `side_to_move`, and `ply` |
| **Rewards** | Sparse: +1 win, -1 loss, 0 otherwise |
| **VariantConfig** | Customize rules without code changes; frozen for thread safety |
| **FEN** | Text notation for any position; useful for curriculum learning |
| **Agents** | Implement `select_move()` to create custom AI players |
| **Gymnasium** | Standard RL interface: obs (14×10×9), actions (0–8279) |

---

## ⏭️ What's Next?

Now that you understand the game and environment, the next tutorials will cover:

- **Tutorial 02: Search Algorithms** — How AlphaBeta search works under the hood, and why it's good but limited
- **Tutorial 03: Monte Carlo Tree Search (MCTS)** — The heart of AlphaZero, built from scratch with visualizations
- **Tutorial 04: AlphaZero Training** — Neural network training, self-play, and how AI bootstraps its own knowledge
- **Tutorial 05: Experiments** — Reward shaping, custom architectures, variant ablation studies

Or if you're impatient, start training right now:

```bash
python -m hybrid train --iterations 10 --games 50 --simulations 50
```